In [ ]:
!pip install google-play-scraper pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.0 MB/s eta 0:00:00


 5 categories (education, ai, entertainment, music, health and fitness), each with a 100,000-record target (500,000 ÷ 5).

 Apps with fewer than 3,000 reviews are discarded, but anything between 3,000 and 10,000 is kept as-is rather than requiring the full 10,000.

 The category loop doesn't stop at a fixed app count. It keeps pulling candidates from the expanded search-term lists until that category's 100,000-record target is reached.  So it can attain records of more than 10 apps if needed.

In [ ]:
import csv
import os
import time
from datetime import datetime

from google_play_scraper import Sort, app as gp_app, reviews as gp_reviews, search as gp_search

output_file = 'PlayStoreReviewsMultipleCategoires.csv'

min_reviews_per_app = 3000
max_reviews_per_app = 10000
min_apps_per_category = 10
max_total_records = 500000
batch_size = 200
request_delay_seconds = 1.5

category_search_terms = {
    'education': [
        'language learning app',
        'online courses',
        'flashcards study app',
        'virtual classroom',
        'learn to code',
        'exam prep app',
        'kids learning app',
        'homework helper',
        'test prep app',
        'skill building app',
        'study planner app',
        'reading practice app',
        'math practice app',
        'vocabulary builder app',
        'certification exam prep',
    ],
    'ai': [
        'ai chatbot',
        'ai keyboard',
        'ai photo assistant',
        'ai homework helper',
        'ai math solver',
        'ai writing assistant',
        'ai voice assistant',
        'ai image generator',
        'ai companion app',
        'ai translator',
        'ai productivity assistant',
        'ai note taking app',
        'ai resume builder',
        'ai avatar generator',
        'ai search assistant',
    ],
    'entertainment': [
        'movies and tv shows streaming',
        'video sharing app',
        'live streaming app',
        'short video app',
        'live tv app',
        'comics reader app',
        'anime streaming app',
        'game streaming app',
        'podcast app',
        'ebook reader app',
        'movie ticket booking app',
        'celebrity news app',
        'meme app',
        'trivia game app',
        'social video app',
    ],
    'music': [
        'music streaming',
        'song identifier',
        'internet radio',
        'music player',
        'karaoke app',
        'music download app',
        'lyrics app',
        'dj mixing app',
        'guitar tuner app',
        'music production app',
        'podcast and music app',
        'audiobook app',
        'sound recording app',
        'beat maker app',
        'concert tickets app',
    ],
    'health and fitness': [
        'calorie tracker',
        'fitness tracker app',
        'meditation app',
        'workout planner',
        'sleep tracker',
        'step counter app',
        'yoga app',
        'running app',
        'water reminder app',
        'period tracker app',
        'weight loss app',
        'home workout app',
        'mental health app',
        'diet planner app',
        'gym workout log app',
    ],
}

fieldnames = [
    'app_id',
    'app_name',
    'app_category',
    'app_aggregate_rating',
    'review_text',
    'review_rating',
    'helpful_votes',
    'review_date',
    'scraped_at',
]


def candidate_app_ids_for_category(search_terms, used_app_ids):
    for search_term in search_terms:
        try:
            results = gp_search(search_term, lang='en', country='us', n_hits=10)
        except Exception as error:
            print('search failed for term', search_term, error)
            continue

        for candidate in results:
            candidate_id = candidate.get('appId')

            if candidate_id and candidate_id not in used_app_ids:
                used_app_ids.add(candidate_id)
                yield candidate_id


def scrape_app_reviews(app_id, app_name, category, aggregate_rating, target):
    rows = []
    continuation_token = None

    while len(rows) < target:
        count_this_call = min(batch_size, target - len(rows))

        result, continuation_token = gp_reviews(
            app_id,
            lang='en',
            country='us',
            sort=Sort.NEWEST,
            count=count_this_call,
            continuation_token=continuation_token,
        )

        if not result:
            break

        for entry in result:
            rows.append({
                'app_id': app_id,
                'app_name': app_name,
                'app_category': category,
                'app_aggregate_rating': aggregate_rating,
                'review_text': entry.get('content', ''),
                'review_rating': entry.get('score', ''),
                'helpful_votes': entry.get('thumbsUpCount', 0),
                'review_date': entry.get('at', ''),
                'scraped_at': datetime.now().isoformat(),
            })

        time.sleep(request_delay_seconds)

        if continuation_token is None:
            break

    return rows


def main():
    categories_target_records = max_total_records // len(category_search_terms)
    used_app_ids = set()

    if os.path.exists(output_file):
        os.remove(output_file)

    total_records = 0

    with open(output_file, 'w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()

        for category, search_terms in category_search_terms.items():
            print('scraping category:', category)

            category_records = 0
            category_apps_count = 0

            for app_id in candidate_app_ids_for_category(search_terms, used_app_ids):
                if total_records >= max_total_records:
                    print('reached overall record limit')
                    return

                if category_records >= categories_target_records:
                    break

                try:
                    details = gp_app(app_id)
                except Exception as error:
                    print('skipping', app_id, error)
                    continue

                app_name = details.get('title', app_id)
                aggregate_rating = details.get('score', '')

                remaining_category_budget = categories_target_records - category_records
                remaining_overall_budget = max_total_records - total_records

                target = min(
                    max_reviews_per_app,
                    remaining_category_budget,
                    remaining_overall_budget,
                )

                rows = scrape_app_reviews(
                    app_id,
                    app_name,
                    category,
                    aggregate_rating,
                    target,
                )

                if len(rows) < min_reviews_per_app:
                    print('skipping', app_name, 'only', len(rows), 'reviews')
                    continue

                for row in rows:
                    writer.writerow(row)

                handle.flush()

                total_records += len(rows)
                category_records += len(rows)
                category_apps_count += 1

                print('scraped', len(rows), 'reviews for', app_name)

            if category_apps_count < min_apps_per_category:
                print('warning:', category, 'only', category_apps_count, 'qualifying apps')


if __name__ == '__main__':
    main()

scraping category: education
scraped 10000 reviews for Duolingo: Language Lessons
scraped 10000 reviews for Airlearn - Language Learning
scraped 10000 reviews for Falou - Fast language learning
scraped 10000 reviews for Babbel - Learn Languages
scraped 7114 reviews for Speak: Language Learning
scraped 4847 reviews for Ling - Language Learning
scraped 10000 reviews for LingoDeer - Learn Languages
scraped 8062 reviews for Mango Languages Learning
skipping YayTalk: Speak & Learn English only 66 reviews
skipping LinDuo - Speak Languages only 2 reviews
scraped 10000 reviews for Udemy - Online Courses
scraped 10000 reviews for Coursera: Grow your career
scraped 10000 reviews for edX online learning
scraping category: ai
scraped 10000 reviews for PolyBuzz: Chat with AI Friends
scraped 10000 reviews for AI Chatbot - Nova
scraped 10000 reviews for Talkie: Creative AI Community
scraped 10000 reviews for Character AI: Chat, Talk, Text
scraped 10000 reviews for ChatGPT
scraped 10000 reviews for Ch

In [ ]:
import pandas as pd

df = pd.read_csv('PlayStoreReviewsMultipleCategoires.csv')

total_apps = df['app_id'].nunique()
total_categories = df['app_category'].nunique()
avg_reviews_per_app = df.groupby('app_id').size().mean()
avg_reviews_per_category = df.groupby('app_category').size().mean()

print('total apps in dataset:', total_apps)
print('total categories in dataset:', total_categories)
print('average reviews per app:', round(avg_reviews_per_app, 2))
print('average reviews per category:', round(avg_reviews_per_category, 2))

total apps in dataset: 54
total categories in dataset: 5
average reviews per app: 9261.76
average reviews per category: 100027.0


In [ ]:
from google.colab import files

df = pd.read_csv('PlayStoreReviewsMultipleCategoires.csv')
df.to_csv('PlayStoreReviewsMultipleCategoires.csv', index=False)

files.download('PlayStoreReviewsMultipleCategoires.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>